# Tests: `fasterai.quantize.quantize_callback` (source `nbs/quantize/quantize_callback.ipynb`)

In [ ]:
from fastcore.test import *
import copy
import warnings
import torch
import torch.nn as nn
from fastai.callback.core import Callback
from fasterai.quantize.quantize_callback import *
from fasterai.quantize.quantize_callback import _batch_sizes, _check_static_batch
from fasterai.quantize.quantizer import Quantizer, _HAS_PT2E, _prepare_pt2e

In [ ]:
from fastcore.test import *

# Construction with defaults
cb = QuantizeCallback()
test_eq(cb.backend, 'x86')
test_eq(cb.use_per_tensor, False)
test_eq(cb.verbose, False)
assert cb.quantizer is None  # created lazily in before_fit
assert cb.original_model is None

# Construction with custom params
cb2 = QuantizeCallback(backend='qnnpack', use_per_tensor=True, verbose=True)
test_eq(cb2.backend, 'qnnpack')
test_eq(cb2.use_per_tensor, True)
test_eq(cb2.verbose, True)

# Construction with pre-built quantizer
from fasterai.quantize.quantizer import Quantizer
q = Quantizer(backend='x86', method='qat')
cb3 = QuantizeCallback(quantizer=q)
test_eq(cb3.quantizer.method, 'qat')

# The pt2e backend is a construction away, and its QAT precision is the symmetric INT8 one
from fasterai.core.precision import _resolve_spec
from fasterai.quantize.quantizer import _HAS_PT2E

cb4 = QuantizeCallback(backend='pt2e')
test_eq(cb4.backend, 'pt2e')
assert cb4.quantizer is None  # the quantizer is built in `before_fit`, not here
# The precision is pure python, so it resolves the same way on a runner whose torch build ships no
# pt2e kernels: these assertions run everywhere, and the `Quantizer` one below where the kernels are.
_pt2e_spec = _resolve_spec('pt2e', 'qat')
test_eq((_pt2e_spec.label, _pt2e_spec.qscheme, _pt2e_spec.symmetric), ('W8A8', 'per_channel', True))
test_eq(_pt2e_spec.exports, True)  # a QDQ ONNX export can write what this QAT produces
if _HAS_PT2E:
    test_eq(Quantizer(backend='pt2e', method='qat').spec, _pt2e_spec)
else:
    # ...and without those kernels, asking for the backend fails loudly, naming what is missing
    with ExceptionExpected(ImportError, regex="quantize_pt2e"):
        Quantizer(backend='pt2e', method='qat')

# --- one batch size, and the callback says so before a fit rather than during one ---
class _FakeDL:
    def __init__(self, n, bs, drop_last=False): self.n, self.bs, self.drop_last = n, bs, drop_last
class _FakeDLs:
    def __init__(self, train, valid): self.train, self.valid = train, valid

test_eq(_batch_sizes(_FakeDL(48, 16)), {16})               # 3 full batches
test_eq(_batch_sizes(_FakeDL(50, 16)), {16, 2})            # ...and a short one
test_eq(_batch_sizes(_FakeDL(50, 16, drop_last=True)), {16})
test_eq(_batch_sizes(_FakeDL(12, 16)), {12})               # fewer items than one batch: only a short one
test_eq(_batch_sizes(_FakeDL(12, 16, drop_last=True)), set())  # ...and this loader drops it: no batch at all
test_eq(_batch_sizes(None), set())                         # nothing known, nothing claimed
test_eq(_batch_sizes(_FakeDL(0, 16)), set())

_check_static_batch(_FakeDLs(_FakeDL(48, 16, drop_last=True), _FakeDL(16, 16)))  # uniform: accepted
with ExceptionExpected(ValueError, regex="one batch size"):
    _check_static_batch(_FakeDLs(_FakeDL(48, 16, drop_last=True), _FakeDL(12, 16)))
with ExceptionExpected(ValueError, regex="one batch size"):  # same length, different batch size
    _check_static_batch(_FakeDLs(_FakeDL(48, 16, drop_last=True), _FakeDL(48, 8)))

# --- arguments the precision grammar cannot honor are refused, and say which flow reads them ---
# (these need no `_HAS_PT2E` guard either: the grammar is resolved before the backend is looked for)
with ExceptionExpected(ValueError, regex="legacy-backend flag"):
    Quantizer(backend='pt2e', method='qat', use_per_tensor=True)
with ExceptionExpected(TypeError, regex="`backend` must be a str"):
    Quantizer(backend=8, method='qat')
# `Quantizer.quantize` is the post-training flow: pt2e QAT is this callback's job, and it says so
if _HAS_PT2E:
    with ExceptionExpected(ValueError, regex="QuantizeCallback"):
        Quantizer(backend='pt2e', method='qat').quantize(nn.Linear(4, 4), [torch.randn(2, 4)])

In [ ]:
# --- the optimizer follows the model this callback swaps in ---
from torch.utils.data import TensorDataset
from fastai.data.core import DataLoaders
from fastai.learner import Learner
from fasterai.quantize.quantizer import _HAS_PT2E

def _tiny_model():
    "Conv-BN-ReLU-Pool-Linear, small enough to prepare and train inside a test"
    torch.manual_seed(0)
    return nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(8, 10))

def _tiny_dls(n_train=32, n_valid=16, bs=8):
    "Synthetic dataloaders whose batches all have the same size — the one an exported graph runs"
    torch.manual_seed(0)
    X, y = torch.randn(n_train + n_valid, 3, 8, 8), torch.randint(0, 10, (n_train + n_valid,))
    return DataLoaders.from_dsets(TensorDataset(X[:n_train], y[:n_train]),
                                  TensorDataset(X[n_train:], y[n_train:]), bs=bs, device='cpu')

def _marked(opt, flag='do_wd', value=False):
    "Parameter TENSORS `opt` marked with `flag` — preparation renames parameters, it does not move them"
    return {id(p) for p, state in opt.state.items() if state.get(flag) is value}

class _Probe(Callback):
    "Reads what the optimizer holds once the QAT callback has swapped the model in"
    order = 99  # after QuantizeCallback, so it sees the prepared model
    def before_fit(self):
        self.prepared, self.moved = self.learn.model, []
        self.before = {n: p.detach().clone() for n, p in self.prepared.named_parameters()}
        held = {id(p) for group in self.learn.opt.param_lists for p in group}
        self.untracked = [n for n, p in self.prepared.named_parameters()
                          if p.requires_grad and id(p) not in held]
        self.hypers = [dict(h) for h in self.learn.opt.hypers]
        self.no_wd, self.force_train = _marked(self.learn.opt), _marked(self.learn.opt, 'force_train', True)
    def after_epoch(self):
        # while the model being read is still the prepared one: conversion replaces it in `after_fit`
        self.moved = [n for n, p in self.prepared.named_parameters()
                      if not torch.equal(p.detach(), self.before[n])]

class _EagerProbe(Callback):
    "Reads the optimizer `fit` built, before the QAT callback replaces the model"
    order = -5  # before QuantizeCallback
    def before_fit(self):
        self.no_wd, self.force_train = _marked(self.learn.opt), _marked(self.learn.opt, 'force_train', True)

# FX flow: the parameters the optimizer holds are the ones the prepared model trains, and the
# hyper-parameters `fit` was called with survive the rebuild
_probe, _eager = _Probe(), _EagerProbe()
_learn = Learner(_tiny_dls(), _tiny_model(), loss_func=nn.CrossEntropyLoss(),
                 cbs=[QuantizeCallback(backend='x86', use_per_tensor=True), _probe, _eager])
_learn.fit(2, lr=0.0123, wd=0.037)
test_eq(_probe.untracked, [])
assert _probe.moved, "no parameter of the prepared model moved: QAT trained something else"
test_eq(_probe.hypers[0]['lr'], 0.0123)
test_eq(_probe.hypers[0]['wd'], 0.037)
assert hasattr(_learn, 'quantized_model')
# ...and so do the per-parameter marks: fastai finds the norm and bias parameters by walking the
# MODEL, which preparation rewrites — the conv bias, both batch-norm parameters and the head bias
test_eq(len(_eager.no_wd), 4)
test_eq(_probe.no_wd, _eager.no_wd)                # weight decay still skips the same four tensors
test_eq(_probe.force_train, _eager.force_train)    # ...and the same two stay trainable when frozen
# the FX flow records the precision it applied, the same way the post-training paths do
from fasterai.core.precision import quant_spec
test_eq((quant_spec(_learn.model).backend, quant_spec(_learn.model).method), ('x86', 'qat'))
test_eq(quant_spec(_learn.model).exports, False)  # affine activations: not a portable Q/DQ graph

# pt2e flow: the same contract, on a graph captured by torch.export
if _HAS_PT2E:
    _probe2, _eager2 = _Probe(), _EagerProbe()
    _learn2 = Learner(_tiny_dls(), _tiny_model(), loss_func=nn.CrossEntropyLoss(),
                      cbs=[QuantizeCallback(backend='pt2e'), _probe2, _eager2])
    _learn2.fit(2, lr=0.0123, wd=0.037)
    test_eq(_probe2.untracked, [])
    assert _probe2.moved, "no parameter of the prepared graph moved: QAT trained something else"
    test_eq(_probe2.hypers[0]['lr'], 0.0123)
    test_eq(_probe2.hypers[0]['wd'], 0.037)
    # An exported graph holds no `nn.BatchNorm2d` and no `.bias` attribute, so a plain rebuild would
    # put weight decay back on the batch-norm scale. The marks are carried over by tensor identity.
    test_eq(_probe2.no_wd, _eager2.no_wd)
    test_eq(_probe2.force_train, _eager2.force_train)

In [ ]:
# --- the rebind has power: a real configuration where it is what makes QAT train at all ---
# `Quantizer(qconfig_mapping=...)` is public, and torch's learnable fake-quant observes with
# PARAMETERS — its scale and zero-point are trained. The prepared model then holds tensors the
# optimizer `fit` built never saw: measured on this net, 6 trainable parameters become 18.
from torch.ao.quantization import QConfig, QConfigMapping
from torch.ao.quantization.observer import MovingAverageMinMaxObserver
from torch.ao.quantization._learnable_fake_quantize import _LearnableFakeQuantize

def _learnable_mapping():
    "A qconfig mapping whose fake-quantizers train their own scale and zero-point"
    def _fq(**kwargs):
        return _LearnableFakeQuantize.with_args(observer=MovingAverageMinMaxObserver,
                                                scale=1.0, zero_point=0.0, **kwargs)
    return QConfigMapping().set_global(QConfig(
        activation=_fq(quant_min=0, quant_max=255, dtype=torch.quint8, qscheme=torch.per_tensor_affine),
        weight=_fq(quant_min=-128, quant_max=127, dtype=torch.qint8, qscheme=torch.per_tensor_symmetric)))

_learnable = _Probe()
Learner(_tiny_dls(), _tiny_model(), loss_func=nn.CrossEntropyLoss(),
        cbs=[QuantizeCallback(quantizer=Quantizer(method='qat', qconfig_mapping=_learnable_mapping())),
             _learnable]).fit(2, lr=0.01)
test_eq(_learnable.untracked, [])   # the optimizer holds the parameters the preparation created...
_trained_scales = [n for n in _learnable.moved if n.endswith(('.scale', '.zero_point'))]
assert len(_trained_scales) >= 8, f"the fake-quant parameters were not trained: {_trained_scales}"

# The synthetic version of the same failure, for the flows that DO reuse their source tensors: under
# torch 2.9.1 both preparation passes hand back modules that keep them, which is what makes the
# optimizer `fit` built still usable there. `_Copying` stands in for a pass that allocates its own.
class _Copying(QuantizeCallback):
    "A QAT callback whose preparation hands back a module with fresh parameter tensors"
    def _start_fx(self, example_input):
        super()._start_fx(example_input)
        self._swap_model(copy.deepcopy(self.learn.model))

class _NoRebind(_Copying):
    "...with the optimizer rebind taken out"
    def _rebind_opt(self): pass

_frozen = _Probe()
Learner(_tiny_dls(), _tiny_model(), loss_func=nn.CrossEntropyLoss(),
        cbs=[_NoRebind(backend='x86', use_per_tensor=True), _frozen]).fit(1)
test_eq(_frozen.moved, [])   # the optimizer updated another module: QAT observed frozen weights
assert _frozen.untracked, "the optimizer should hold none of the prepared model's parameters here"

_rebound = _Probe()
Learner(_tiny_dls(), _tiny_model(), loss_func=nn.CrossEntropyLoss(),
        cbs=[_Copying(backend='x86', use_per_tensor=True), _rebound]).fit(1)
assert _rebound.moved, "the rebind is what makes the very same fit train the model that is quantized"
test_eq(_rebound.untracked, [])

# --- a splitter written for the source model ---
from fastai.torch_core import params
if _HAS_PT2E:
    def _brittle(model):
        "Splits the eager model, and cannot split the prepared one"
        if isinstance(model, nn.Sequential): return [params(model)]
        raise TypeError("this splitter only knows the eager model")

    # it still holds every parameter being trained, so the fit goes on — and says so
    with warnings.catch_warnings(record=True) as _caught:
        warnings.simplefilter('always')
        Learner(_tiny_dls(), _tiny_model(), loss_func=nn.CrossEntropyLoss(), splitter=_brittle,
                cbs=[QuantizeCallback(backend='pt2e')]).fit(1)
    assert any('could not be rebuilt' in str(w.message) for w in _caught), [str(w.message) for w in _caught]

    # ...but a splitter that also leaves parameters out of it would train a lie, and is refused
    with ExceptionExpected(RuntimeError, regex="untrained"):
        Learner(_tiny_dls(), _tiny_model(), loss_func=nn.CrossEntropyLoss(),
                splitter=lambda m: [params(m[0])],
                cbs=[QuantizeCallback(backend='pt2e')]).fit(1)

In [ ]:
# --- pt2e QAT end to end: train, convert, export the trained INT8 graph ---
import numpy as np
import tempfile
from pathlib import Path
from fasterai.core.precision import quant_spec
from fasterai.export.onnx_exporter import ONNXModel, _has_package, export_qdq, qdq_stats

# The same export refuses the FX-QAT model of the cell above, reading the spec that flow recorded.
# No guard: `export_qdq` reads that spec before it looks for onnx, or for a pt2e graph to write.
with ExceptionExpected(ValueError, regex="export_qdq cannot write"):
    export_qdq(_learn.model, torch.randn(8, 3, 8, 8), Path(tempfile.gettempdir())/'never_written.onnx')

if _HAS_PT2E:
    _qat_learn = Learner(_tiny_dls(), _tiny_model(), loss_func=nn.CrossEntropyLoss(),
                         cbs=[QuantizeCallback(backend='pt2e', verbose=True)])
    _qat_learn.fit(2, lr=0.01)

    _sample = torch.randn(8, 3, 8, 8)   # the batch size the graph was captured with
    _qat_out = _qat_learn.model(_sample)
    test_eq(_qat_out.shape, (8, 10))
    assert torch.isfinite(_qat_out).all()
    assert _qat_learn.model is _qat_learn.quantized_model

    # the precision that ran travels with the model, and an exporter can read it
    _spec = quant_spec(_qat_learn.model)
    test_eq((_spec.backend, _spec.method, _spec.label), ('pt2e', 'qat', 'W8A8'))
    test_eq((_spec.qscheme, _spec.symmetric, _spec.exports), ('per_channel', True, True))
    # symmetric INT8: every zero-point in the converted graph is 0
    assert all(int(b.abs().max()) == 0 for n, b in _qat_learn.model.named_buffers() if 'zero_point' in n)
    # the converted graph refuses the eager `.train()`/`.eval()`, so the callback bound them to the
    # rewrite torch does instead — without it the fastai loop cannot run a single validation
    _qat_learn.model.eval(); _qat_learn.model.train(); _qat_learn.model.eval()
    assert torch.isfinite(_qat_learn.model(_sample)).all()

    if all(_has_package(p) for p in ('onnx', 'onnxscript', 'onnxruntime')):
        with tempfile.TemporaryDirectory() as _tmp:
            _qat_path = export_qdq(_qat_learn.model, _sample, Path(_tmp)/'qat_qdq.onnx')
            _qat_stats = qdq_stats(_qat_path)
            assert _qat_stats.n_quantize > 0 and _qat_stats.n_dequantize > 0, _qat_stats
            assert _qat_stats.n_per_channel > 0, "per-channel weights did not survive the export"
            test_eq(_qat_stats.n_nonzero_zero_point, 0)  # portability: zero_point == 0 everywhere

            # ...and the graph that came out answers what the trained model answers. Measured over 18
            # QAT runs (torch 2.9.1, onnxruntime CPU, this net, 2 epochs, one training-batch order
            # each): max|Δlogit| on the SAME batch stayed <= 9.2e-3 — ONNX Runtime runs real INT8
            # kernels, so it is not bit-exact — against >= 1.0e-1 when the ONNX arm is fed a DIFFERENT
            # batch; an independent re-measurement saw that same-batch spread reach 9.4e-3 and the
            # different-batch minimum 8.6e-2. `_QAT_ATOL` sits between the two with ~3x headroom on
            # each side, and the last assertion is what proves the bracket is not vacuous.
            _QAT_ATOL = 3e-2
            _session = ONNXModel(_qat_path)
            with torch.no_grad(): _pt_logits = _qat_learn.model(_sample).numpy()
            _onnx_logits = _session(_sample).numpy()
            assert np.allclose(_pt_logits, _onnx_logits, atol=_QAT_ATOL), \
                f"max|Δlogit| = {np.abs(_pt_logits - _onnx_logits).max():.3e}"
            _other = torch.randn(8, 3, 8, 8, generator=torch.Generator().manual_seed(1))
            assert not np.allclose(_pt_logits, _session(_other).numpy(), atol=_QAT_ATOL), \
                "the parity check cannot tell one input batch from another — it proves nothing"

# --- pt2e QAT failure modes are loud ---
if _HAS_PT2E:
    class _DataDependent(nn.Module):
        "Its control flow depends on the data, so torch.export cannot capture it"
        def __init__(self):
            super().__init__()
            self.fc = nn.Linear(3 * 8 * 8, 10)
        def forward(self, x):
            x = x.flatten(1)
            if x.sum() > 0: return self.fc(x) * 2
            return self.fc(x)

    try:
        Learner(_tiny_dls(), _DataDependent(), loss_func=nn.CrossEntropyLoss(),
                cbs=[QuantizeCallback(backend='pt2e')]).fit(1)
        raise AssertionError("an uncapturable model must raise instead of training in floating point")
    except RuntimeError as e:
        assert "torch.export" in str(e), f"unhelpful message: {e}"
        assert e.__cause__ is not None, "the original diagnostic must be chained"

    # a validation loader whose last batch is shorter is refused BEFORE the capture, rather than by a
    # shape guard in the middle of the fit it invalidates
    with ExceptionExpected(ValueError, regex="one batch size"):
        Learner(_tiny_dls(n_train=32, n_valid=12), _tiny_model(), loss_func=nn.CrossEntropyLoss(),
                cbs=[QuantizeCallback(backend='pt2e')]).fit(1)
else:
    # On a torch build without the pt2e kernels, the fit stops in `before_fit` with the ImportError
    # that names them — it never trains a model in floating point and calls the result quantized.
    with ExceptionExpected(ImportError, regex="quantize_pt2e"):
        Learner(_tiny_dls(), _tiny_model(), loss_func=nn.CrossEntropyLoss(),
                cbs=[QuantizeCallback(backend='pt2e')]).fit(1)

# --- QAT quantizes DURING training, it does not merely watch ---
if _HAS_PT2E:
    # A QAT preparation inserts fake-quantize modules where post-training quantization inserts plain
    # observers. The difference is visible in the very first forward pass: the prepared graph rounds,
    # so it does not answer exactly what the float model it was built from answers.
    _fq_cb = QuantizeCallback(backend='pt2e')

    class _FakeQuantProbe(Callback):
        "Compares the prepared graph with the float model it was built from, before any training"
        order = 99
        def before_fit(self):
            self.x = torch.randn(8, 3, 8, 8)
            self.quantized = self.learn.model.eval()(self.x).detach()
            self.float = _fq_cb.original_model.eval()(self.x).detach()

    _fq_probe = _FakeQuantProbe()
    Learner(_tiny_dls(), _tiny_model(), loss_func=nn.CrossEntropyLoss(),
            cbs=[_fq_cb, _fq_probe]).fit(1)
    _delta = (_fq_probe.quantized - _fq_probe.float).abs().max()
    # The floor this pin has to clear: the same preparation asked for the POST-TRAINING spec only
    # observes, so its graph answers what the float model answers, to the float noise between an eager
    # model and its captured graph. Measured (single run, torch 2.9.1 CPU, n=12 successive input
    # draws): QAT 4.4e-3 to 3.5e-2 — it grows because the moving-average observers keep updating —
    # against a floor of 3.0e-8 to 6.0e-8, on logits reaching 4.4e-1. The 1e-4 pin sits between the
    # two regimes, and the floor assertion is what keeps it from going stale green.
    _observed = _prepare_pt2e(_fq_cb.original_model, _fq_probe.x, _resolve_spec('pt2e'))
    _floor = (_observed(_fq_probe.x).detach() - _fq_probe.float).abs().max()
    assert _floor < 1e-6, f"the observe-only floor moved ({_floor:.1e}): the pin below means nothing"
    assert _delta > 1e-4, f"the prepared graph answers what the float model answers ({_delta:.1e}): " \
                          "it is watching the training, not quantizing it"
    assert _delta < 0.1, f"the prepared graph is a different model, not a rounded one ({_delta:.1e})"

In [ ]:
#| slow
# Full QAT training loop — verify model is prepared and converted
import torch.nn as nn
from torch.utils.data import TensorDataset
from fastai.data.core import DataLoaders
from fastai.learner import Learner

_model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10)
)
_X = torch.randn(64, 3, 8, 8)
_y = torch.randint(0, 10, (64,))
_dls = DataLoaders.from_dsets(
    TensorDataset(_X[:48], _y[:48]),
    TensorDataset(_X[48:], _y[48:]),
    bs=16, device='cpu'
)

_cb = QuantizeCallback(backend='x86', use_per_tensor=True)
_learn = Learner(_dls, _model, loss_func=nn.CrossEntropyLoss(), cbs=[_cb])
_learn.fit(2)

# Verify QAT produced a quantized model
assert hasattr(_learn, 'quantized_model'), "quantized_model should be set by after_fit"
assert _cb.original_model is not None, "original model backup should exist"

# Verify the quantized model can still produce output
with torch.no_grad():
    _out = _learn.quantized_model(_X[:1].cpu())
test_eq(_out.shape, (1, 10))

In [ ]:
#| slow
# Integration test: pt2e QAT on a real ResNet-18, exported to a QDQ ONNX graph
if _HAS_PT2E:
    from torchvision.models import resnet18

    torch.manual_seed(0)
    _rn_X, _rn_y = torch.randn(24, 3, 64, 64), torch.randint(0, 10, (24,))
    _rn_dls = DataLoaders.from_dsets(TensorDataset(_rn_X[:16], _rn_y[:16]),
                                     TensorDataset(_rn_X[16:], _rn_y[16:]), bs=8, device='cpu')
    _rn_probe = _Probe()
    _rn_learn = Learner(_rn_dls, resnet18(weights=None, num_classes=10),
                        loss_func=nn.CrossEntropyLoss(),
                        cbs=[QuantizeCallback(backend='pt2e'), _rn_probe])
    _rn_learn.fit(2, lr=1e-3, wd=0.037)

    # the optimizer trained the graph that was quantized, with the hyper-parameters `fit` was given
    test_eq(_rn_probe.untracked, [])
    assert len(_rn_probe.moved) > 50, f"only {len(_rn_probe.moved)} parameters moved"
    test_eq(_rn_probe.hypers[0]['lr'], 1e-3)
    test_eq(_rn_probe.hypers[0]['wd'], 0.037)

    _rn_sample = torch.randn(8, 3, 64, 64)
    _rn_out = _rn_learn.model(_rn_sample)
    test_eq(_rn_out.shape, (8, 10))
    assert torch.isfinite(_rn_out).all()
    assert all(int(b.abs().max()) == 0 for n, b in _rn_learn.model.named_buffers() if 'zero_point' in n)
    test_eq(quant_spec(_rn_learn.model).method, 'qat')

    if all(_has_package(p) for p in ('onnx', 'onnxscript')):
        with tempfile.TemporaryDirectory() as _tmp:
            _rn_path = export_qdq(_rn_learn.model, _rn_sample, Path(_tmp)/'resnet18_qat_qdq.onnx')
            _rn_stats = qdq_stats(_rn_path)
            # one per-channel weight tensor per conv (20) plus the classifier
            test_eq(_rn_stats.n_per_channel, 21)
            assert _rn_stats.n_quantize > 20, _rn_stats
            test_eq(_rn_stats.n_nonzero_zero_point, 0)

In [ ]:
#| slow
# Integration test: pt2e QAT with a Q/DQ placement, through a real fastai training loop
import contextlib
import io

if _HAS_PT2E:
    class _ResidualNet(nn.Module):
        "One residual block, so that `qdq_placement='skip_conv_add'` has an edge to act on"
        def __init__(self, n_classes=10):
            super().__init__()
            self.stem = nn.Conv2d(3, 8, 3, padding=1)
            self.branch = nn.Sequential(nn.Conv2d(8, 8, 3, padding=1), nn.BatchNorm2d(8))
            self.head = nn.Sequential(nn.Flatten(), nn.Linear(8 * 8 * 8, n_classes))
        def forward(self, x):
            x = torch.relu(self.stem(x))
            return self.head(torch.relu(x + self.branch(x)))

    torch.manual_seed(0)
    # `Quantizer(backend='pt2e', method='qat', qdq_placement=...)` is the route to a QAT placement,
    # and `method='qat'` is not optional: a quantizer handed to this callback keeps the method it was
    # built with, so one built without it would run the POST-TRAINING preparation inside the fit.
    _pl_quantizer = Quantizer(backend='pt2e', method='qat', qdq_placement='skip_conv_add')
    test_eq((_pl_quantizer.spec.method, _pl_quantizer.spec.qdq_placement), ('qat', 'skip_conv_add'))
    _pl_learn = Learner(_tiny_dls(), _ResidualNet(), loss_func=nn.CrossEntropyLoss(),
                        cbs=[QuantizeCallback(quantizer=_pl_quantizer, verbose=True)])
    _pl_log = io.StringIO()
    with contextlib.redirect_stdout(_pl_log): _pl_learn.fit(2, lr=0.01)
    _pl_log = _pl_log.getvalue()
    # `verbose` reaches the placement pass: a QAT run is the one path where the count of cleared
    # edges cannot be read any other way before the training is over
    assert "left 1 of 1 addition(s)" in _pl_log, _pl_log
    assert "qdq_placement='skip_conv_add'" in _pl_log, _pl_log
    # ...and the line that reports the prepared precision names the placement beside it
    assert "prepared for QAT (W8A8, per_channel, qdq_placement=skip_conv_add)" in _pl_log, _pl_log

    _pl_sample = torch.randn(8, 3, 8, 8)   # the batch size the graph was captured with
    test_eq(_pl_learn.model(_pl_sample).shape, (8, 10))
    assert torch.isfinite(_pl_learn.model(_pl_sample)).all()
    assert _pl_learn.model is _pl_learn.quantized_model
    # the placement travels with the trained model, beside the precision that ran
    test_eq(quant_spec(_pl_learn.model).qdq_placement, 'skip_conv_add')
    test_eq((quant_spec(_pl_learn.model).method, quant_spec(_pl_learn.model).label), ('qat', 'W8A8'))
    assert all(int(b.abs().max()) == 0
               for n, b in _pl_learn.model.named_buffers() if 'zero_point' in n)

    # ...and it is in the FILE the training produced. QAT annotates BEFORE conv+bn fusion, so the
    # partition the pass matched here ended at a batch-norm rather than at the convolution itself —
    # this is the assertion that pins that the QAT graph shape is one the matcher handles.
    if all(_has_package(p) for p in ('onnx', 'onnxscript')):
        with tempfile.TemporaryDirectory() as _tmp:
            _pl_path = export_qdq(_pl_learn.model, _pl_sample, Path(_tmp)/'qat_placement.onnx')
            _pl_stats = qdq_stats(_pl_path)
            test_eq(_pl_stats.n_unquantized_conv_add, 1)
            test_eq(_pl_stats.n_nonzero_zero_point, 0)
            assert _pl_stats.n_per_channel > 0, "per-channel weights did not survive the export"

    # the same option on a model with no residual addition refuses, inside `before_fit`, rather than
    # training a model whose provenance would claim a placement that never ran
    with ExceptionExpected(ValueError, regex="found no conv"):
        Learner(_tiny_dls(), _tiny_model(), loss_func=nn.CrossEntropyLoss(),
                cbs=[QuantizeCallback(quantizer=Quantizer(backend='pt2e', method='qat',
                                                          qdq_placement='skip_conv_add'))]).fit(1)